In [7]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkb, wkt
from libpysal.weights import DistanceBand
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [6]:
gdf = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
gdf.head()

,NUTS_CODE,NUTS_NAME,RecordType,Total_cases2,Forest,Shrub,Urban,Crop,Water,Other,...,gdp,incidence,pop_dens,forest2,shrub2,crop2,urban2,water2,other2,geometry
0,AE,None,DENGUE,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON EMPTY
1,AE_MV_SA,None,ZIKV,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON EMPTY
2,AF,None,DENGUE,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON EMPTY
3,AG,None,CHIK,12.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON EMPTY
4,AG,None,DENGUE,8.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MULTIPOLYGON EMPTY


In [8]:
# ---------------------------
# 1) Filter to valid WNV rows
# ---------------------------
gdf = gdf.loc[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()
if "RecordType" in gdf.columns:
    gdf = gdf.loc[gdf["RecordType"] != "ZIKV"].copy()

# ---------------------------
# 2) Keep columns & aggregate
# ---------------------------
avg_var = [
    "Forest", "Shrub", "Urban", "Crop", "Water", "Other",
    "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
    "Mean_T_Spring", "Max_DD_Spring",
    # "Mean_T_Summer",  # include if you want it
    "Max_WD_Autumn", "Max_DD_Autumn",
    "gdp", "pop_dens"
]
y_var = "incidence"
id_cols = ["NUTS_NAME","NUTS_CODE"]

Q:\UserTemp\massaro\AppData\Local\Temp\113\ipykernel_31916\2586838586.py:4: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  gdf = gdf.loc[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()


In [9]:
keep = [c for c in ([y_var] + avg_var + id_cols + ["geometry"]) if c in gdf.columns]
gdf = gdf[keep].copy()

# Sum incidence, mean predictors at NUTS3
agg = {y_var: "sum"} | {v: "mean" for v in avg_var if v in gdf.columns}
fin = (gdf.groupby(id_cols, as_index=False).agg(agg))

# Reattach one geometry per NUTS3
geom_first = gdf[id_cols + ["geometry"]].drop_duplicates(id_cols)
fin = fin.merge(geom_first, on=id_cols, how="left")
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=gdf.crs)

# Drop NA on variables we need
needed = [y_var] + [v for v in avg_var if v in fin.columns]
fin = fin.dropna(subset=needed).copy()


In [11]:
from libpysal.weights import DistanceBand, lag_spatial

In [12]:
# ---------------------------
# 3) Standardize (R-style: ddof=1)
# ---------------------------
num_cols = fin.select_dtypes(include="number").columns.tolist()
for c in num_cols:
    s = fin[c].astype(float)
    fin[c] = (s - s.mean()) / s.std(ddof=1)

# ---------------------------
# 4) Project to metric & Wy
# ---------------------------
fin = fin.to_crs(32633)  # UTM 33N
cent = fin.geometry.centroid
coords = np.c_[cent.x.values, cent.y.values]

# Distance band (choose one):
max_distance = 150_000  # or 150_000, per your sensitivity pick
# A) Row-standardized average (common):
w = DistanceBand.from_array(coords, threshold=max_distance, binary=True, silence_warnings=True)
w.transform = "R"
Wy = lag_spatial(w, fin[y_var].to_numpy())

In [13]:

fin["Y_weighted"] = Wy

# ---------------------------
# 5) OLS (baseline on RAW incidence)  -> this matches LM in your Table 1
# ---------------------------
X_ols = fin[avg_var].drop(columns=[c for c in ["Mean_T_Summer"] if c in avg_var], errors="ignore")
y_ols = fin[y_var]
X_ols = sm.add_constant(X_ols, has_constant="add")
ols_res = sm.OLS(y_ols, X_ols).fit()

ols_tbl = pd.DataFrame({
    "Covariate": X_ols.columns,
    "Estimate": np.round(ols_res.params.values, 2),
    "SE": np.round(ols_res.bse.values, 2),
    "p-value": [("<.001" if p < 0.001 else f"{p:.3f}") for p in ols_res.pvalues.values]
})


In [14]:
ols_tbl 

,Covariate,Estimate,SE,p-value
0,const,0.00,0.03,1.000
1,Forest,-0.12,0.04,0.003
2,Shrub,0.12,0.03,<.001
3,Urban,-0.01,0.03,0.843
4,Crop,-0.04,0.04,0.369
5,Water,0.07,0.03,0.032
6,Other,0.02,0.03,0.490
7,Mean_P_Winter,0.01,0.05,0.825
8,Max_WD_Winter,-0.01,0.06,0.873
9,Max_DD_Winter,0.03,0.05,0.582


In [18]:
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

In [20]:
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
# Replico i filtri: geometrie non vuote e RecordType != "ZIKV"
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

# ------------------------------------------------------------------
# 3) PREPARA DATASET come in R (selezione variabili e aggregazione)
# ------------------------------------------------------------------
avg_var = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + avg_var +
             ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()

# In R raggruppi per geometry, NUTS_NAME, NUTS_CODE; in GeoPandas
# conviene usare l'identificativo NUTS_CODE (che è unico) e conservare la prima geometria.
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in avg_var})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg_dict)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)

# Centroidi per controllo/coords (come in R)
fin["cent"] = fin.geometry.centroid

# ------------------------------------------------------------------
# 4) STANDARDIZZA (z-score) tutte le variabili numeriche
# ------------------------------------------------------------------
num_cols = fin.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
fin_scaled = fin.copy()
fin_scaled[num_cols] = scaler.fit_transform(fin_scaled[num_cols])

# ------------------------------------------------------------------
# 5) RIPROIEZIONE a metrica (EPSG:32633 ~ UTM 33N), come in R
# ------------------------------------------------------------------
fin_m = fin_scaled.to_crs(32633)

# ------------------------------------------------------------------
# 6) COSTRUISCI matrici di vicinato entro 150 km e Y laggata
# ------------------------------------------------------------------
# In R, coordinates() sullo Spatial di poligoni restituisce i centroidi.
# Qui uso i centroidi in metri (proiezione UTM 33N).
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

# threshold 150 km, binaria, senza diagonale
W = DistanceBand(coords, threshold=150000, binary=True, silence_warnings=True)
# trasformo in matrice sparsa CSR (se serve), o applico direttamente come W * y
# libpysal conserva W in forma weights + neighbors; uso full() per matrice densa se serve
W_sparse = W.sparse  # scipy CSR matrix

# y laggata
y = fin_m["incidence"].to_numpy().reshape(-1, 1)
y_lag = W_sparse.dot(y).ravel()
fin_m["Y_weighted"] = y_lag

# ------------------------------------------------------------------
# 7) STIMA OLS con formula analoga all’R (Mean_T_Summer esclusa come nel codice R)
# ------------------------------------------------------------------
# NOTA: qui stai regredendo Y_lag su X (non è una SAR/SLX); è OLS classico.
formula = ("Y_weighted ~ Forest + Shrub + Urban + Crop + Water + Other + "
           "Mean_P_Winter + Max_WD_Winter + Max_DD_Winter + "
           "Mean_T_Spring + Max_DD_Spring + "
           # "Mean_T_Summer + "  # esclusa come in R
           "Max_WD_Autumn + Max_DD_Autumn + gdp + pop_dens")

# statsmodels richiede un DataFrame "piatto" (niente geometry)
Xy = pd.DataFrame(fin_m.drop(columns=["geometry", "cent"]))
lm_fit = smf.ols(formula, data=Xy).fit()

# ------------------------------------------------------------------
# 8) VIF per multicollinearità globale (come car::vif)
# ------------------------------------------------------------------
# Creo la matrice di design senza l'intercetta per VIF
design_cols = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
               "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
               "Mean_T_Spring", "Max_DD_Spring",
               "Max_WD_Autumn", "Max_DD_Autumn",
               "gdp", "pop_dens"]
X_for_vif = Xy[design_cols].copy()
X_for_vif = sm.add_constant(X_for_vif, has_constant="add")  # per sicurezza



# ------------------------------------------------------------------
# 9) Tabella coefficienti formattata come in R
# ------------------------------------------------------------------
coefs = lm_fit.summary2().tables[1].reset_index().rename(columns={"index": "term"})
coefs["estimate"] = coefs["Coef."].round(2)
coefs["std.error"] = coefs["Std.Err."].round(2)

def fmt_p(p):
    try:
        return "<.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return str(p)

coefs["p-value"] = coefs["P>|t|"].apply(fmt_p)

lm_table = coefs.loc[:, ["term", "estimate", "std.error", "p-value"]]
lm_table = lm_table.rename(columns={
    "term": "Covariate",
    "estimate": "Estimate",
    "std.error": "SE"
})

print("\n=== Coefficient table ===")
print(lm_table.to_string(index=False))
# Se vuoi salvare:

Q:\UserTemp\massaro\AppData\Local\Temp\113\ipykernel_31916\2793943224.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fin["cent"] = fin.geometry.centroid



=== Coefficient table ===
    Covariate  Estimate   SE p-value
    Intercept     -4.12 0.25   <.001
       Forest      0.19 0.34   0.581
        Shrub      1.52 0.29   <.001
        Urban      0.76 0.27   0.004
         Crop      0.57 0.33   0.091
        Water      0.76 0.27   0.005
        Other      0.54 0.30   0.069
Mean_P_Winter      3.94 0.39   <.001
Max_WD_Winter     -2.42 0.52   <.001
Max_DD_Winter      0.09 0.41   0.819
Mean_T_Spring      2.94 0.45   <.001
Max_DD_Spring     -2.98 0.37   <.001
Max_WD_Autumn     -1.43 0.43   <.001
Max_DD_Autumn      0.94 0.45   0.036
          gdp     -2.48 0.32   <.001
     pop_dens      1.04 0.27   <.001


In [21]:
vif_table = pd.DataFrame({
    "variable": design_cols,
    "VIF": [variance_inflation_factor(X_for_vif.values, i+1)  # +1 per saltare la costante
            for i in range(len(design_cols))]
}).sort_values("VIF", ascending=False)

print("\n=== VIF (global multicollinearity) ===")
print(vif_table.to_string(index=False))

MissingDataError: exog contains inf or nans